## Assignment 2
### Question 1 : Fisher Iris's Flowers

This is a very famous dataset collected by Fisher Iris way back in 1930s, it has the Petal,sepal widths and lengths corresponding to 4 different flower species. Make a MLP using NN such that it classifies these 4 flower species.

In [148]:
# getting dataset
import kagglehub
target_dir = "./dataset"
path = kagglehub.dataset_download("uciml/iris", output_dir=target_dir)
print("Path to dataset files:", path)

Path to dataset files: ./dataset


In [149]:
#importing libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as f
import torchvision.transforms as transforms
import pandas as pd
import sklearn.model_selection as ms
from torch.utils.data import TensorDataset, DataLoader

In [150]:
# Creating Neural Network using nn

class NN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NN, self).__init__()
        self.fc1 = nn.Linear(input_size,16)
        self.fc2 = nn.Linear(16,num_classes)

    def forward(self,x):
        x = f.relu(self.fc1(x))
        x= self.fc2(x)
        return x

In [151]:
# setting up device
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [152]:
# parameters
input_size = 4
num_classes = 3
learning_rate = 0.001
batch_size = 30
num_epochs = 100

In [153]:
df = pd.read_csv('./dataset/Iris.csv')
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [154]:
label_map={
    "Iris-setosa":0,
    "Iris-versicolor":1,
    "Iris-virginica":2
}

X = df.iloc[:,1:-1]
Y = df.iloc[:,-1]

train_X, test_X, train_Y, test_Y = ms.train_test_split(X,Y, test_size=0.2, random_state=42)

train_X = torch.tensor(train_X.values,dtype=torch.float32)
test_X = torch.tensor(test_X.values,dtype=torch.float32)

train_Y = train_Y.map(label_map)
test_Y = test_Y.map(label_map)

train_Y = torch.tensor(train_Y.values,dtype=torch.long)
test_Y = torch.tensor(test_Y.values,dtype=torch.long)

train_dataset = TensorDataset(train_X,train_Y)
test_dataset = TensorDataset(test_X,test_Y)

train_loader = DataLoader(train_dataset, batch_size=30, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=30)

In [155]:
# initialize model
model = NN(input_size=input_size,num_classes=num_classes).to(device=device)

# Loss and optimize
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate)

In [156]:
print(train_loader.batch_size)

30


In [157]:
# training the netwrok
for epoch in range(num_epochs):

    running_loss = 0.0

    for idx,(data,targets) in enumerate(train_loader):
        # dataframes to tensor
        data=data.to(device=device)
        targets = targets.to(device=device)
        # This step is important as this ensures all data is transferred to memory of same device

        # forward
        results = model(data)
        loss = criterion(results, targets)

        # back propogation
        optimizer.zero_grad() # gradients are accumulated, so this clears old gradients and fresh gradients are calculated
        loss.backward() # Calculates the gradients for this step
        optimizer.step() # Applies new gradients

        running_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {running_loss/len(train_loader):.4f}")
    

Epoch 1: Loss = 1.8067
Epoch 2: Loss = 1.6969
Epoch 3: Loss = 1.5846


Epoch 4: Loss = 1.4923
Epoch 5: Loss = 1.3871
Epoch 6: Loss = 1.3068
Epoch 7: Loss = 1.2256
Epoch 8: Loss = 1.1504
Epoch 9: Loss = 1.0751
Epoch 10: Loss = 1.0254
Epoch 11: Loss = 0.9760
Epoch 12: Loss = 0.9414
Epoch 13: Loss = 0.9043
Epoch 14: Loss = 0.8832
Epoch 15: Loss = 0.8639
Epoch 16: Loss = 0.8526
Epoch 17: Loss = 0.8393
Epoch 18: Loss = 0.8303
Epoch 19: Loss = 0.8200
Epoch 20: Loss = 0.8108
Epoch 21: Loss = 0.8009
Epoch 22: Loss = 0.7909
Epoch 23: Loss = 0.7809
Epoch 24: Loss = 0.7708
Epoch 25: Loss = 0.7602
Epoch 26: Loss = 0.7498
Epoch 27: Loss = 0.7393
Epoch 28: Loss = 0.7300
Epoch 29: Loss = 0.7184
Epoch 30: Loss = 0.7082
Epoch 31: Loss = 0.6981
Epoch 32: Loss = 0.6882
Epoch 33: Loss = 0.6792
Epoch 34: Loss = 0.6699
Epoch 35: Loss = 0.6604
Epoch 36: Loss = 0.6517
Epoch 37: Loss = 0.6431
Epoch 38: Loss = 0.6352
Epoch 39: Loss = 0.6266
Epoch 40: Loss = 0.6188
Epoch 41: Loss = 0.6110
Epoch 42: Loss = 0.6035
Epoch 43: Loss = 0.5963
Epoch 44: Loss = 0.5889
Epoch 45: Loss = 0.582

In [158]:
# Evaluating the results

def check_accu(loader, model):
    num_correct = 0
    num_total = 0

    model.eval() # Puts the model into evaluation mode, weights are not updated

    with torch.no_grad():
        for x,y in loader:
            x = x.to(device = device)
            y = y.to(device = device)

            results = model(x)
            _, pred = torch.max(results, 1)
            
            num_total += y.size(0)
            num_correct += (pred == y).sum().item()
        
        print(f"Accuracy = {100*num_correct/num_total:.2f}%")

check_accu(test_loader, model)

Accuracy = 96.67%


In [170]:
# User testing

def predict_flower(sepal_length, sepal_width, petal_length, petal_width):

    sample = torch.tensor([[sepal_length, sepal_width, petal_length, petal_width]], dtype=torch.float32)
    sample = sample.to(device)
    
    model.eval()
    with torch.no_grad():
        result = model(sample)
        _,pred = torch.max(result,1)

        for k,v in label_map.items():
            if v==pred.item():
                print(f"Prediction : {k}")
                break


predict_flower(4.8,3.4,1.6,0.2)
predict_flower(6.1,2.9,4.7,1.4)
predict_flower(5.9,3.0,5.1,1.8)


Prediction : Iris-setosa
Prediction : Iris-versicolor
Prediction : Iris-virginica
